<a href="https://colab.research.google.com/github/CodeCrusaders26/privacy-dns-filter/blob/feature%2Fml-model/notebooks/processed_dataset_v2_csv.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Task 1: Inspect the Original Dataset

We will inspect `/content/combined_domains.csv` to look at:
- Dataset shape, column names, and data types.
- The first 10 rows to understand the representation of domains, sources, and labels.
- Missing values and duplicate domain entries.

In [ ]:
import pandas as pd

# Load the dataset
file_path = '/content/combined_domains.csv'
df = pd.read_csv(file_path)

# Display shape and basic info
print("=== Dataset Shape ===")
print(df.shape)

print("\n=== Data Types and Info ===")
print(df.info())

print("\n=== Missing Values ===")
print(df.isnull().sum())

print("\n=== First 10 Rows ===")
display(df.head(10))

# Identify unique counts for domain and source/label columns
print("\n=== Column Value Counts & Statistics ===")
for col in df.columns:
    unique_count = df[col].nunique()
    print(f"Column '{col}' has {unique_count} unique values.")
    if unique_count < 20:
        print(df[col].value_counts(dropna=False))

# Check for duplicate domains
if 'domain' in df.columns:
    duplicate_count = df['domain'].duplicated().sum()
    print(f"\nDuplicate domains in dataset: {duplicate_count}")
elif 'Domain' in df.columns:
    duplicate_count = df['Domain'].duplicated().sum()
    print(f"\nDuplicate domains in dataset: {duplicate_count}")
else:
    # Fallback to check duplicates for the first column if no named domain column is detected
    duplicate_count = df.iloc[:, 0].duplicated().sum()
    print(f"\nDuplicate entries in the first column: {duplicate_count}")

=== Dataset Shape ===
(19950, 4)

=== Data Types and Info ===
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19950 entries, 0 to 19949
Data columns (total 4 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   domain           19950 non-null  object
 1   label            19950 non-null  int64 
 2   source           19950 non-null  object
 3   collection_date  19950 non-null  object
dtypes: int64(1), object(3)
memory usage: 623.6+ KB
None

=== Missing Values ===
domain             0
label              0
source             0
collection_date    0
dtype: int64

=== First 10 Rows ===


,domain,label,source,collection_date
0,findmykids.org,0,Tranco,2026-11-11
1,labs.google,0,Tranco,2026-11-11
2,integrating.smbc-flexibler.com,1,PhishTank,2026-11-11
3,unl.edu,0,Tranco,2026-11-11
4,ggi.dietondemand.com,1,URLhaus,2026-11-11
5,bnf.fr,0,Tranco,2026-11-11
6,qualys.com,0,Tranco,2026-11-11
7,t.yesware.com,1,PhishTank,2026-11-11
8,tagesschau.de,0,Tranco,2026-11-11
9,flashscore.ua,0,Tranco,2026-11-11



=== Column Value Counts & Statistics ===
Column 'domain' has 19950 unique values.
Column 'label' has 2 unique values.
label
1    10000
0     9950
Name: count, dtype: int64
Column 'source' has 4 unique values.
source
Tranco               9950
PhishTank            9016
URLhaus               976
PhishTank+URLhaus       8
Name: count, dtype: int64
Column 'collection_date' has 1 unique values.
collection_date
2026-11-11    19950
Name: count, dtype: int64

Duplicate domains in dataset: 0


### Task 2.1: Source-Label Validation & Normalization Impact Check

We will check for domain normalization issues and cross-reference sources against labels to ensure perfect logical consistency before altering any data.

In [ ]:
import pandas as pd

# Ensure we are using the loaded df
# 1. Cross-tabulation of source vs label
print("=== Source vs Label Cross-tabulation ===")
cross_tab = pd.crosstab(df['source'], df['label'])
display(cross_tab)

# 2. Inspect the 8 records with source 'PhishTank+URLhaus'
print("\n=== Records from PhishTank+URLhaus ===")
duall_source_df = df[df['source'] == 'PhishTank+URLhaus']
display(duall_source_df)

# 3. Test impact of normalization on domain uniqueness
# We will test lowercasing, stripping whitespace, and removing trailing dots (if any).
normalized_series = df['domain'].astype(str).str.lower().str.strip().str.rstrip('.')
original_unique_count = df['domain'].nunique()
normalized_unique_count = normalized_series.nunique()

print(f"\n=== Normalization Impact Analysis ===")
print(f"Original unique domains count: {original_unique_count}")
print(f"Normalized unique domains count: {normalized_unique_count}")

if original_unique_count != normalized_unique_count:
    print("\nWARNING: Normalization will merge some domain strings! Let's find them:")
    # Identify which domains are causing conflicts or becoming duplicates
    temp_df = df.copy()
    temp_df['normalized_domain'] = normalized_series
    dup_normalized = temp_df[temp_df['normalized_domain'].duplicated(keep=False)].sort_values(by='normalized_domain')
    display(dup_normalized.head(20))
else:
    print("\nExcellent: Domain normalization will not create any duplicate domain records.")

=== Source vs Label Cross-tabulation ===


label,0,1
source,,
PhishTank,0,9016
PhishTank+URLhaus,0,8
Tranco,9950,0
URLhaus,0,976



=== Records from PhishTank+URLhaus ===


,domain,label,source,collection_date
1947,j-0-09g-9bh-h-ggf-1324992141.cos.ap-bangkok.my...,1,PhishTank+URLhaus,2026-11-11
2965,drive.google.com,1,PhishTank+URLhaus,2026-11-11
3396,firebasestorage.googleapis.com,1,PhishTank+URLhaus,2026-11-11
7107,f3i5-0g49bgn-3h95-1324992141.cos.ap-jakarta.my...,1,PhishTank+URLhaus,2026-11-11
7373,www.dropbox.com,1,PhishTank+URLhaus,2026-11-11
9107,docs.google.com,1,PhishTank+URLhaus,2026-11-11
12945,bj5y6-0f-9h4-9fgg4-1324992141.cos.ap-bangkok.m...,1,PhishTank+URLhaus,2026-11-11
18238,linktr.ee,1,PhishTank+URLhaus,2026-11-11



=== Normalization Impact Analysis ===
Original unique domains count: 19950
Normalized unique domains count: 19950

Excellent: Domain normalization will not create any duplicate domain records.


### Task 2.2: Locate and Inspect Original Source Datasets

We will search for raw original files containing datasets from Tranco, URLhaus, and PhishTank in `/content/` and look for any Google Drive mounts or local CSVs.

In [1]:
import os
import glob
import pandas as pd

# Let's search the workspace and display paths
candidate_files = glob.glob('/content/**/*', recursive=True) + glob.glob('/content/drive/MyDrive/**/*', recursive=True)

source_keywords = ['tranco', 'urlhaus', 'phishtank', 'top-1m', 'verified_online']
found_files = []

for file_path in candidate_files:
    if os.path.isfile(file_path):
        lower_name = os.path.basename(file_path).lower()
        if any(keyword in lower_name for keyword in source_keywords):
            found_files.append(file_path)

print("=== Found Source Datasets ===")
if not found_files:
    print("No matching raw source files found.")
else:
    for f in sorted(list(set(found_files))):
        print(f)

# Let's inspect columns for the files that are present locally in `/content/` to identify URL/domain columns
for path in ['/content/top-1m.csv', '/content/verified_online.csv', '/content/csv.txt']:
    if os.path.exists(path):
        print(f"\n=== Inspecting: {path} ===")
        try:
            # Read top few lines to detect columns without parsing entire large files
            temp_df = pd.read_csv(path, nrows=5)
            print(f"Columns: {list(temp_df.columns)}")
            display(temp_df.head(2))
        except Exception as e:
            print(f"Could not preview: {e}")

=== Found Source Datasets ===
/content/top-1m.csv
/content/verified_online.csv

=== Inspecting: /content/top-1m.csv ===
Columns: ['1', 'google.com']


,1,google.com
0,2,cloudflare.com
1,3,ezviz7.com



=== Inspecting: /content/verified_online.csv ===
Columns: ['phish_id', 'url', 'phish_detail_url', 'submission_time', 'verified', 'verification_time', 'online', 'target']


,phish_id,url,phish_detail_url,submission_time,verified,verification_time,online,target
0,9557831,https://qlw.dnx.mybluehost.me/arubamain/jS/,http://www.phishtank.com/phish_detail.php?phis...,2026-11-11T07:01:44+00:00,yes,2026-11-11T07:11:37+00:00,yes,Other
1,9557822,https://webmaillll-login.duckdns.org,http://www.phishtank.com/phish_detail.php?phis...,2026-11-11T06:00:14+00:00,yes,2026-11-11T06:11:29+00:00,yes,Other



=== Inspecting: /content/csv.txt ===
Columns: ['################################################################']


,################################################################
0,# abuse.ch URLhaus Database Dump (CSV) ...
1,# Last updated: 2026-11-11 08:52:08 (UTC) ...


### Task 1: Correctly load Tranco
We load `top-1m.csv` with designated headers, check its size, and display the first 10 rows.

In [2]:
import pandas as pd

tranco_path = '/content/top-1m.csv'
tranco_df = pd.read_csv(tranco_path, names=['rank', 'domain'])
print(f"Tranco dataset loaded. Total domains: {len(tranco_df)}")
display(tranco_df.head(10))

Tranco dataset loaded. Total domains: 1000000


,rank,domain
0,1,google.com
1,2,cloudflare.com
2,3,ezviz7.com
3,4,amazonaws.com
4,5,facebook.com
5,6,fbcdn.net
6,7,hicloudcam.com
7,8,googleapis.com
8,9,gstatic.com
9,10,youtube.com


### Task 2 & 3: Load PhishTank and URLhaus, Extract Hostnames
We will load PhishTank and extract hostnames using `urllib.parse`. Then we will load URLhaus `/content/csv.txt` (skipping lines starting with `#`), identify the URL column, and extract hostnames.

In [5]:
from urllib.parse import urlparse
import pandas as pd
import numpy as np

# Helper function to reliably extract domain/hostnames
def extract_hostname(url_str):
    try:
        if not url_str.startswith(('http://', 'https://')):
            url_str = 'http://' + str(url_str)
        parsed = urlparse(url_str)
        host = parsed.hostname
        return host if host else ""
    except:
        return ""

# 1. Load PhishTank
phishtank_path = '/content/verified_online.csv'
phishtank_df = pd.read_csv(phishtank_path)
phishtank_df['extracted_host'] = phishtank_df['url'].apply(extract_hostname)
print(f"PhishTank Loaded: {len(phishtank_df)} records.")

# 2. Load URLhaus skipping lines with tokenizing errors gracefully
urlhaus_path = '/content/csv.txt'
urlhaus_df = pd.read_csv(
    urlhaus_path,
    comment='#',
    names=['id', 'dateadded', 'url', 'url_status', 'threat', 'tags', 'urlhaus_link', 'reporter'],
    quoting=3,
    escapechar='\\',
    on_bad_lines='skip'
)
urlhaus_df['extracted_host'] = urlhaus_df['url'].apply(extract_hostname)
print(f"URLhaus Loaded: {len(urlhaus_df)} records.")

PhishTank Loaded: 46808 records.
URLhaus Loaded: 27343 records.


### Task 4 & 5: Investigate the 8 domains and create the report
We inspect each of the 8 target domains across the original sources and generate our summary table.

In [6]:
target_domains = [
    "j-0-09g-9bh-h-ggf-1324992141.cos.ap-bangkok.myqcloud.com",
    "drive.google.com",
    "firebasestorage.googleapis.com",
    "f3i5-0g49bgn-3h95-1324992141.cos.ap-jakarta.myqcloud.com",
    "www.dropbox.com",
    "docs.google.com",
    "bj5y6-0f-9h4-9fgg4-1324992141.cos.ap-bangkok.myqcloud.com",
    "linktr.ee"
]

results = []

for domain in target_domains:
    pt_matches = phishtank_df[phishtank_df['extracted_host'] == domain]
    in_pt = len(pt_matches) > 0
    pt_urls = pt_matches['url'].tolist() if in_pt else []

    uh_matches = urlhaus_df[urlhaus_df['extracted_host'] == domain]
    in_uh = len(uh_matches) > 0
    uh_urls = uh_matches['url'].tolist() if in_uh else []

    orig_url = "N/A"
    if in_pt:
        orig_url = pt_urls[0]
    elif in_uh:
        orig_url = uh_urls[0]

    is_legit_platform = domain in [
        "drive.google.com",
        "firebasestorage.googleapis.com",
        "www.dropbox.com",
        "docs.google.com",
        "linktr.ee"
    ]

    evidence = ""
    if in_pt or in_uh:
        if is_legit_platform:
            evidence = "Malicious URL hosted on a legitimate platform"
        else:
            evidence = "Potentially malicious domain (Cos/Cloud storage domain or threat host)"
    else:
        evidence = "Unverified / No Match in original sources"

    results.append({
        "Domain": domain,
        "In PhishTank": "Yes" if in_pt else "No",
        "In URLhaus": "Yes" if in_uh else "No",
        "Original URL Match (Example)": orig_url[:80] + "..." if len(orig_url) > 80 else orig_url,
        "Evidence / Classification": evidence
    })

report_df = pd.DataFrame(results)
display(report_df)

,Domain,In PhishTank,In URLhaus,Original URL Match (Example),Evidence / Classification
0,j-0-09g-9bh-h-ggf-1324992141.cos.ap-bangkok.my...,Yes,No,https://j-0-09g-9bh-h-ggf-1324992141.cos.ap-ba...,Potentially malicious domain (Cos/Cloud storag...
1,drive.google.com,Yes,No,https://drive.google.com/file/d/17GAdRTuFpI-wO...,Malicious URL hosted on a legitimate platform
2,firebasestorage.googleapis.com,Yes,No,https://firebasestorage.googleapis.com/v0/b/en...,Malicious URL hosted on a legitimate platform
3,f3i5-0g49bgn-3h95-1324992141.cos.ap-jakarta.my...,Yes,No,https://f3i5-0g49bgn-3h95-1324992141.cos.ap-ja...,Potentially malicious domain (Cos/Cloud storag...
4,www.dropbox.com,Yes,No,https://www.dropbox.com/scl/fi/up0upd7891g06kb...,Malicious URL hosted on a legitimate platform
5,docs.google.com,Yes,No,https://docs.google.com/presentation/d/e/2PACX...,Malicious URL hosted on a legitimate platform
6,bj5y6-0f-9h4-9fgg4-1324992141.cos.ap-bangkok.m...,Yes,No,http://bj5y6-0f-9h4-9fgg4-1324992141.cos.ap-ba...,Potentially malicious domain (Cos/Cloud storag...
7,linktr.ee,Yes,No,https://linktr.ee/xfkm001,Malicious URL hosted on a legitimate platform


### Detailed Investigation of the 8 Questionable Domains

We will inspect the full untruncated records from PhishTank and the original combined dataset for all 8 suspicious domains.

In [8]:
import pandas as pd
from urllib.parse import urlparse
pd.set_option('display.max_colwidth', None)

# Ensure combined dataset and original source files are loaded
combined_path = '/content/combined_domains.csv'
df = pd.read_csv(combined_path)

target_domains = [
    "j-0-09g-9bh-h-ggf-1324992141.cos.ap-bangkok.myqcloud.com",
    "drive.google.com",
    "firebasestorage.googleapis.com",
    "f3i5-0g49bgn-3h95-1324992141.cos.ap-jakarta.myqcloud.com",
    "www.dropbox.com",
    "docs.google.com",
    "bj5y6-0f-9h4-9fgg4-1324992141.cos.ap-bangkok.myqcloud.com",
    "linktr.ee"
]

detailed_results = []

for domain in target_domains:
    # 1. Look up in original combined dataset
    comb_match = df[df['domain'] == domain]
    current_label = comb_match['label'].values[0] if not comb_match.empty else "Unknown"
    current_source = comb_match['source'].values[0] if not comb_match.empty else "Unknown"

    # 2. Look up matching PhishTank records
    pt_matches = phishtank_df[phishtank_df['extracted_host'] == domain]

    if not pt_matches.empty:
        first_match = pt_matches.iloc[0]
        full_url = first_match['url']
        phish_id = first_match['phish_id']
        verified = first_match['verified']
        online = first_match['online']
        sub_time = first_match['submission_time']
        target = first_match['target']

        hostname_verified = "Yes (Matches Domain)"

        is_legit = domain in [
            "drive.google.com",
            "firebasestorage.googleapis.com",
            "www.dropbox.com",
            "docs.google.com",
            "linktr.ee"
        ]

        if is_legit:
            host_type = "Legitimate service hosting a potentially malicious page"
            appropriate_labeling = "No (Blocking this domain causes extreme false positive/collateral damage)"
            recommendation = "Reconsider label (Recommend label 0 or exclude from training)"
        else:
            host_type = "Dedicated subdomain that may be malicious (Isolated campaign bucket)"
            appropriate_labeling = "Yes (Isolated object storage buckets can be blocked safely)"
            recommendation = "Retain label 1"

        has_path = "Yes" if len(urlparse(full_url).path) > 1 else "No"
    else:
        full_url = "N/A"
        phish_id = "N/A"
        verified = "N/A"
        online = "N/A"
        sub_time = "N/A"
        target = "N/A"
        hostname_verified = "N/A"
        host_type = "Uncertain"
        appropriate_labeling = "N/A"
        recommendation = "Investigate further"
        has_path = "N/A"

    detailed_results.append({
        "Domain": domain,
        "Current Combined Label": current_label,
        "Current Combined Source": current_source,
        "Phish ID": phish_id,
        "Verified": verified,
        "Online": online,
        "Submission Time": sub_time,
        "Target": target,
        "Full Matching URL": full_url,
        "Host Type": host_type,
        "Has Path to Malicious Resource": has_path,
        "Appropriate for Domain Classifier?": appropriate_labeling,
        "Recommendation": recommendation
    })

analysis_df = pd.DataFrame(detailed_results)
display(analysis_df)

,Domain,Current Combined Label,Current Combined Source,Phish ID,Verified,Online,Submission Time,Target,Full Matching URL,Host Type,Has Path to Malicious Resource,Appropriate for Domain Classifier?,Recommendation
0,j-0-09g-9bh-h-ggf-1324992141.cos.ap-bangkok.myqcloud.com,1,PhishTank+URLhaus,8522762,yes,yes,2024-04-05T14:49:22+00:00,Other,https://j-0-09g-9bh-h-ggf-1324992141.cos.ap-bangkok.myqcloud.com/35-0350gh9v-39yh5g.html,Dedicated subdomain that may be malicious (Isolated campaign bucket),Yes,Yes (Isolated object storage buckets can be blocked safely),Retain label 1
1,drive.google.com,1,PhishTank+URLhaus,9319050,yes,yes,2026-01-23T17:29:38+00:00,Internal Revenue Service,https://drive.google.com/file/d/17GAdRTuFpI-wO_v0zDkgg8FhVFCZdiR5/view,Legitimate service hosting a potentially malicious page,Yes,No (Blocking this domain causes extreme false positive/collateral damage),Reconsider label (Recommend label 0 or exclude from training)
2,firebasestorage.googleapis.com,1,PhishTank+URLhaus,9493034,yes,yes,2026-07-31T18:05:24+00:00,Other,https://firebasestorage.googleapis.com/v0/b/encryption-9dcc0.firebasestorage.app/o/Order-encryption-invoice.html?alt=media&amp;token=be7a6c7c-3268-4e4c-aae5-4b5414c4a3e0#,Legitimate service hosting a potentially malicious page,Yes,No (Blocking this domain causes extreme false positive/collateral damage),Reconsider label (Recommend label 0 or exclude from training)
3,f3i5-0g49bgn-3h95-1324992141.cos.ap-jakarta.myqcloud.com,1,PhishTank+URLhaus,8503385,yes,yes,2024-03-22T10:51:40+00:00,Other,https://f3i5-0g49bgn-3h95-1324992141.cos.ap-jakarta.myqcloud.com/36hg-04ik6-9j4-9h5.html,Dedicated subdomain that may be malicious (Isolated campaign bucket),Yes,Yes (Isolated object storage buckets can be blocked safely),Retain label 1
4,www.dropbox.com,1,PhishTank+URLhaus,9557173,yes,yes,2026-11-09T18:39:27+00:00,Other,https://www.dropbox.com/scl/fi/up0upd7891g06kbcy6lc7/ATT-MAILBOX-UPDATE.paper,Legitimate service hosting a potentially malicious page,Yes,No (Blocking this domain causes extreme false positive/collateral damage),Reconsider label (Recommend label 0 or exclude from training)
5,docs.google.com,1,PhishTank+URLhaus,9543256,yes,yes,2026-10-13T10:20:23+00:00,Other,https://docs.google.com/presentation/d/e/2PACX-1vT6_zXw7SXsLYcjfjxWpbiuSY9W6xI8eb6_Gbfw_KsOJ2NvmtxNs7A0s9U3XNYnNBEBkcW5oz-BOQjU/pub?pli=1&amp;slide=id.p,Legitimate service hosting a potentially malicious page,Yes,No (Blocking this domain causes extreme false positive/collateral damage),Reconsider label (Recommend label 0 or exclude from training)
6,bj5y6-0f-9h4-9fgg4-1324992141.cos.ap-bangkok.myqcloud.com,1,PhishTank+URLhaus,8511440,yes,yes,2024-03-31T14:39:09+00:00,Other,http://bj5y6-0f-9h4-9fgg4-1324992141.cos.ap-bangkok.myqcloud.com/yn5og-40i6-9gu-9hjf.html,Dedicated subdomain that may be malicious (Isolated campaign bucket),Yes,Yes (Isolated object storage buckets can be blocked safely),Retain label 1
7,linktr.ee,1,PhishTank+URLhaus,9465487,yes,yes,2026-06-29T03:50:06+00:00,Other,https://linktr.ee/xfkm001,Legitimate service hosting a potentially malicious page,Yes,No (Blocking this domain causes extreme false positive/collateral damage),Reconsider label (Recommend label 0 or exclude from training)


### Task 5: Detailed Recommendation and Risks of URL-to-Domain Threat Mapping

#### Summary Recommendations per Domain
- **j-0-09g-9bh-h-ggf-1324992141.cos.ap-bangkok.myqcloud.com**: **Retain label 1**. This is a dedicated, ephemeral, isolated cloud storage bucket used exclusively for hosting malicious files.
- **f3i5-0g49bgn-3h95-1324992141.cos.ap-jakarta.myqcloud.com**: **Retain label 1**. (Same context as above).
- **bj5y6-0f-9h4-9fgg4-1324992141.cos.ap-bangkok.myqcloud.com**: **Retain label 1**. (Same context as above).
- **drive.google.com**: **Reconsider label**. This is a highly critical domain. Blocking it breaks normal workflow capabilities.
- **docs.google.com**: **Reconsider label**. This is a core Google Suite domain. Blocking it causes massive false positives.
- **firebasestorage.googleapis.com**: **Reconsider label**. Firebase hosts millions of legitimate mobile and web application backend resources.
- **www.dropbox.com**: **Reconsider label**. Popular collaborative file hosting platform; blocking it creates high collateral damage.
- **linktr.ee**: **Reconsider label**. Extremely common social bio-linking tool. Only specific accounts, not the host domain, are malicious.

#### Risks of URL-level Threat Reports as Domain-Level Labels
When using raw URL lists (such as PhishTank or URLhaus) to construct domain-level classifiers, direct truncation to the hostname leads to **Overblocking (False Positives)**.

Security threat intelligence platforms register the complete resource identifier (`https://legitimate.com/malicious/path.html`). Truncating this to `legitimate.com` trains classifiers to block millions of legitimate daily interactions, as the model cannot distinguish between the underlying infrastructure domain and the user-generated content hosted inside the paths.

### Task 2.3: Source-Attribution Audit for Questionable Domains

We will audit the eight questionable domains against the original raw sources (**Tranco**, **PhishTank**, and **URLhaus**) to explain why they are classified under the source `PhishTank+URLhaus` in `combined_domains.csv`.

In [9]:
import pandas as pd
from urllib.parse import urlparse

# Ensure datasets are fully accessible
combined_path = '/content/combined_domains.csv'
df = pd.read_csv(combined_path)

target_domains = [
    "j-0-09g-9bh-h-ggf-1324992141.cos.ap-bangkok.myqcloud.com",
    "drive.google.com",
    "firebasestorage.googleapis.com",
    "f3i5-0g49bgn-3h95-1324992141.cos.ap-jakarta.myqcloud.com",
    "www.dropbox.com",
    "docs.google.com",
    "bj5y6-0f-9h4-9fgg4-1324992141.cos.ap-bangkok.myqcloud.com",
    "linktr.ee"
]

audit_results = []

for domain in target_domains:
    # 1. Check Tranco
    tranco_match = tranco_df[tranco_df['domain'] == domain]
    in_tranco = "Yes" if not tranco_match.empty else "No"

    # 2. Check PhishTank
    pt_match = phishtank_df[phishtank_df['extracted_host'] == domain]
    in_pt = "Yes" if not pt_match.empty else "No"

    # 3. Check URLhaus
    uh_match = urlhaus_df[urlhaus_df['extracted_host'] == domain]
    in_uh = "Yes" if not uh_match.empty else "No"

    # 4. Read the record in combined dataset
    comb_row = df[df['domain'] == domain]
    comb_label = comb_row['label'].values[0] if not comb_row.empty else "N/A"
    comb_src = comb_row['source'].values[0] if not comb_row.empty else "N/A"

    # Let's also check if there is an approximate match in URLhaus (e.g. domain with/without www, or url substring)
    # to understand if a slight normalization mismatch caused the attribution discrepancy
    stripped_domain = domain.replace("www.", "")
    approx_uh_match = urlhaus_df[urlhaus_df['extracted_host'].str.replace("www.", "", regex=False) == stripped_domain]
    approx_pt_match = phishtank_df[phishtank_df['extracted_host'].str.replace("www.", "", regex=False) == stripped_domain]

    in_uh_approx = "Yes" if not approx_uh_match.empty else "No"
    in_pt_approx = "Yes" if not approx_pt_match.empty else "No"

    # Explanation generator
    if in_pt == "Yes" and in_uh == "No":
        if in_uh_approx == "Yes":
            discrepancy = f"Urlhaus match found with domain normalization (e.g., www mismatch)."
            correct_source = "PhishTank+URLhaus"
        else:
            discrepancy = "Assigned 'PhishTank+URLhaus' in combined dataset despite no record in original URLhaus."
            correct_source = "PhishTank"
    else:
        discrepancy = "Perfect match with original labels."
        correct_source = comb_src

    audit_results.append({
        "Domain": domain,
        "Original Combined Label": comb_label,
        "Original Combined Source": comb_src,
        "Exact Tranco Match": in_tranco,
        "Exact PhishTank Match": in_pt,
        "Exact URLhaus Match": in_uh,
        "Approx URLhaus Match (No-WWW)": in_uh_approx,
        "Correct Source Attribution": correct_source,
        "Discrepancy Explanation": discrepancy
    })

audit_df = pd.DataFrame(audit_results)
display(audit_df)

,Domain,Original Combined Label,Original Combined Source,Exact Tranco Match,Exact PhishTank Match,Exact URLhaus Match,Approx URLhaus Match (No-WWW),Correct Source Attribution,Discrepancy Explanation
0,j-0-09g-9bh-h-ggf-1324992141.cos.ap-bangkok.myqcloud.com,1,PhishTank+URLhaus,No,Yes,No,No,PhishTank,Assigned 'PhishTank+URLhaus' in combined dataset despite no record in original URLhaus.
1,drive.google.com,1,PhishTank+URLhaus,No,Yes,No,No,PhishTank,Assigned 'PhishTank+URLhaus' in combined dataset despite no record in original URLhaus.
2,firebasestorage.googleapis.com,1,PhishTank+URLhaus,No,Yes,No,No,PhishTank,Assigned 'PhishTank+URLhaus' in combined dataset despite no record in original URLhaus.
3,f3i5-0g49bgn-3h95-1324992141.cos.ap-jakarta.myqcloud.com,1,PhishTank+URLhaus,No,Yes,No,No,PhishTank,Assigned 'PhishTank+URLhaus' in combined dataset despite no record in original URLhaus.
4,www.dropbox.com,1,PhishTank+URLhaus,No,Yes,No,No,PhishTank,Assigned 'PhishTank+URLhaus' in combined dataset despite no record in original URLhaus.
5,docs.google.com,1,PhishTank+URLhaus,No,Yes,No,No,PhishTank,Assigned 'PhishTank+URLhaus' in combined dataset despite no record in original URLhaus.
6,bj5y6-0f-9h4-9fgg4-1324992141.cos.ap-bangkok.myqcloud.com,1,PhishTank+URLhaus,No,Yes,No,No,PhishTank,Assigned 'PhishTank+URLhaus' in combined dataset despite no record in original URLhaus.
7,linktr.ee,1,PhishTank+URLhaus,Yes,Yes,No,No,PhishTank,Assigned 'PhishTank+URLhaus' in combined dataset despite no record in original URLhaus.


**Reasoning**:
Load the dataset and display the filtered dataframe containing the 8 specific domains of interest to confirm their labels, sources, and metadata as specified in the instructions.



In [10]:
import pandas as pd

# 1. Read the dataset
combined_df = pd.read_csv('/content/combined_domains.csv')

# 2. Print shape and columns
print(f"Dataset Shape: {combined_df.shape}")
print(f"Columns: {list(combined_df.columns)}")

# 3. Filter and display all rows for the 8 specific domains
target_domains = [
    "drive.google.com",
    "docs.google.com",
    "firebasestorage.googleapis.com",
    "www.dropbox.com",
    "linktr.ee",
    "j-0-09g-9bh-h-ggf-1324992141.cos.ap-bangkok.myqcloud.com",
    "f3i5-0g49bgn-3h95-1324992141.cos.ap-jakarta.myqcloud.com",
    "bj5y6-0f-9h4-9fgg4-1324992141.cos.ap-bangkok.myqcloud.com"
]

filtered_df = combined_df[combined_df['domain'].isin(target_domains)]
print("\n=== Target Domains Metadata in Combined Dataset ===")
display(filtered_df)

Dataset Shape: (19950, 4)
Columns: ['domain', 'label', 'source', 'collection_date']

=== Target Domains Metadata in Combined Dataset ===


,domain,label,source,collection_date
1947,j-0-09g-9bh-h-ggf-1324992141.cos.ap-bangkok.myqcloud.com,1,PhishTank+URLhaus,2026-11-11
2965,drive.google.com,1,PhishTank+URLhaus,2026-11-11
3396,firebasestorage.googleapis.com,1,PhishTank+URLhaus,2026-11-11
7107,f3i5-0g49bgn-3h95-1324992141.cos.ap-jakarta.myqcloud.com,1,PhishTank+URLhaus,2026-11-11
7373,www.dropbox.com,1,PhishTank+URLhaus,2026-11-11
9107,docs.google.com,1,PhishTank+URLhaus,2026-11-11
12945,bj5y6-0f-9h4-9fgg4-1324992141.cos.ap-bangkok.myqcloud.com,1,PhishTank+URLhaus,2026-11-11
18238,linktr.ee,1,PhishTank+URLhaus,2026-11-11


# Task
We are ready to clean the dataset, refine the labels, adjust source attributions, and prepare our structured outputs.

Here is our plan to execute these updates safely and systematically:

### Plan of Action:
1. **Audit & Exclusions**:
   * Inspect "/content/combined_domains.csv".
   * Separate the **5 legitimate platform domains** to exclude them from the training set, saving them into an audit file named `excluded_audit_domains.csv`:
     * `drive.google.com`
     * `docs.google.com`
     * `firebasestorage.googleapis.com`
     * `www.dropbox.com`
     * `linktr.ee`
   * Retain the **3 cloud object storage subdomains** in the dataset as malicious (label 1):
     * `j-0-09g-9bh-h-ggf-1324992141.cos.ap-bangkok.myqcloud.com`
     * `f3i5-0g49bgn-3h95-1324992141.cos.ap-jakarta.myqcloud.com`
     * `bj5y6-0f-9h4-9fgg4-1324992141.cos.ap-bangkok.myqcloud.com`

2. **Source Corrections**:
   * For the 3 retained cloud storage domains, update their `source` attribute from `PhishTank+URLhaus` to `PhishTank` since our audit verified they do not exist in the original URLhaus database.

3. **Validation & Distribution Metrics**:
   * Verify the original shape ($19950$ rows) changes to exactly $19945$ rows after the 5 exclusions.
   * Recalculate and output the exact updated benign (0) vs. malicious (1) label distribution.

4. **Preserved Output Files**:
   * Save the refined training dataset to a new safeguarded file: `cleaned_combined_domains.csv`.
   * Save the excluded records to: `excluded_audit_domains.csv`.

Let's proceed with executing this workflow in the next cell.

## Audit Current Dataset and Metadata

### Subtask:
Inspect and audit the original combined domains dataset to verify its exact structure and extract the 8 specific domains of interest.


## Calculate Exclusions and Future Label Distributions

### Subtask:
Compute the exact target shapes, record counts, and updated label distributions after separating the 5 legitimate platform domains.


**Reasoning**:
We will compute the target shapes, record counts, and updated label distributions after separating the 5 legitimate platform domains as instructed.



In [11]:
import pandas as pd

# 1. Define target platform domains to exclude
exclude_domains = [
    'drive.google.com',
    'docs.google.com',
    'firebasestorage.googleapis.com',
    'www.dropbox.com',
    'linktr.ee'
]

# 2. Check original size
orig_size = len(combined_df)
print(f"Original dataset size: {orig_size} records")

# 3. Create cleaned and excluded datasets
cleaned_df = combined_df[~combined_df['domain'].isin(exclude_domains)]
excluded_df = combined_df[combined_df['domain'].isin(exclude_domains)]

# 4. Verify target row counts
new_size = len(cleaned_df)
print(f"Cleaned dataset size: {new_size} records (Expected: 19945)")
print(f"Excluded dataset size: {len(excluded_df)} records (Expected: 5)")
assert new_size == 19945, "Error: Remaining records count is not exactly 19945!"

# 5. Recalculate label distributions
print("\n=== Updated Label Distribution (Cleaned Dataset) ===")
print(cleaned_df['label'].value_counts())
print("\n=== Percentage Distribution ===")
print(cleaned_df['label'].value_counts(normalize=True) * 100)

Original dataset size: 19950 records
Cleaned dataset size: 19945 records (Expected: 19945)
Excluded dataset size: 5 records (Expected: 5)

=== Updated Label Distribution (Cleaned Dataset) ===
label
1    9995
0    9950
Name: count, dtype: int64

=== Percentage Distribution ===
label
1    50.11281
0    49.88719
Name: proportion, dtype: float64


## Correct Source Attributions and Design Preservation Strategy

### Subtask:
Correct the source attribution metadata of the retained cloud storage domains in the cleaned dataset and export both the cleaned dataset and the excluded domains to distinct, safeguarded files.


**Reasoning**:
Update the source of the 3 retained cloud storage domains in cleaned_df from PhishTank+URLhaus to PhishTank, verify the source values, and save both datasets to their respective CSV files.



In [13]:
import pandas as pd

# 1. Locate the 3 retained cloud storage domains in cleaned_df
retained_domains = [
    'j-0-09g-9bh-h-ggf-1324992141.cos.ap-bangkok.myqcloud.com',
    'f3i5-0g49bgn-3h95-1324992141.cos.ap-jakarta.myqcloud.com',
    'bj5y6-0f-9h4-9fgg4-1324992141.cos.ap-bangkok.myqcloud.com'
]

# Copy dataframe to avoid SettingWithCopyWarning
cleaned_df = cleaned_df.copy()

# 2. Update the 'source' attribute for these domains from 'PhishTank+URLhaus' to 'PhishTank'
cleaned_df.loc[cleaned_df['domain'].isin(retained_domains), 'source'] = 'PhishTank'

# 3. Verify that no records in cleaned_df have 'PhishTank+URLhaus' as their source value anymore
remaining_dual_sources = cleaned_df[cleaned_df['source'] == 'PhishTank+URLhaus']
print(f"Number of remaining 'PhishTank+URLhaus' records in cleaned dataset: {len(remaining_dual_sources)}")

print("\n=== Updated Source Distribution ===")
print(cleaned_df['source'].value_counts())

# Verify updated source of the 3 specific domains
print("\n=== Verification of the 3 Cloud Domains ===")
display(cleaned_df[cleaned_df['domain'].isin(retained_domains)])

# 4. Save the modified cleaned_df to a new CSV file
cleaned_df.to_csv('/content/cleaned_combined_domains.csv', index=False)
print("Saved cleaned dataset to /content/cleaned_combined_domains.csv")

# 5. Save the 5 excluded domains to a separate CSV file
excluded_df.to_csv('/content/excluded_audit_domains.csv', index=False)
print("Saved excluded domains to /content/excluded_audit_domains.csv")

Number of remaining 'PhishTank+URLhaus' records in cleaned dataset: 0

=== Updated Source Distribution ===
source
Tranco       9950
PhishTank    9019
URLhaus       976
Name: count, dtype: int64

=== Verification of the 3 Cloud Domains ===


,domain,label,source,collection_date
1947,j-0-09g-9bh-h-ggf-1324992141.cos.ap-bangkok.myqcloud.com,1,PhishTank,2026-11-11
7107,f3i5-0g49bgn-3h95-1324992141.cos.ap-jakarta.myqcloud.com,1,PhishTank,2026-11-11
12945,bj5y6-0f-9h4-9fgg4-1324992141.cos.ap-bangkok.myqcloud.com,1,PhishTank,2026-11-11


Saved cleaned dataset to /content/cleaned_combined_domains.csv
Saved excluded domains to /content/excluded_audit_domains.csv


**Reasoning**:
We will list and inspect the files inside '/content/' and look for any Python scripts or saved configuration files that contain preprocessing, tokenization, or feature engineering logic, and check the columns of the existing dataframes to see if features have already been computed.



In [14]:
import os
import glob
import pandas as pd

# 1. Search for any python files or potential feature extraction scripts in the workspace
py_files = glob.glob('/content/**/*.py', recursive=True)
print("=== Python Files found in workspace ===")
print(py_files)

# 2. Check the columns of the loaded dataframes to see if features were already extracted
print("\n=== Columns in cleaned_df ===")
print(cleaned_df.columns)

# 3. Read the first line of any Python files if found to inspect their purposes
for file_path in py_files:
    print(f"\n--- Previewing: {file_path} ---")
    try:
        with open(file_path, 'r') as f:
            head = [next(f) for _ in range(15)]
            print("".join(head))
    except Exception as e:
        print(f"Could not read file: {e}")

=== Python Files found in workspace ===
[]

=== Columns in cleaned_df ===
Index(['domain', 'label', 'source', 'collection_date'], dtype='object')


## Inspect Preprocessing Pipeline and Feature Extraction Definitions

### Subtask:
Analyze the existing files, variables, and code history to identify the exact feature extraction logic previously applied or defined for the dataset.


## Formulate Reproducible Preprocessing and Validation Workflow

### Subtask:
Define a structured domain preprocessing and feature extraction routine, and compute lexical features for the cleaned dataset.


**Reasoning**:
We will define a robust feature extraction function to compute lexical features from the domains in our cleaned dataset and generate descriptive statistics for validation.



In [19]:
import numpy as np
import pandas as pd
import math
from collections import Counter

# 1. Define feature extraction function
def calculate_entropy(s):
    if not s:
        return 0
    counts = Counter(s)
    length = len(s)
    return -sum((count / length) * math.log2(count / length) for count in counts.values())

def extract_lexical_features(domain):
    if not isinstance(domain, str):
        domain = ""

    domain_len = len(domain)
    dots_count = domain.count('.')
    hyphens_count = domain.count('-')
    digits_count = sum(c.isdigit() for c in domain)
    digit_ratio = (digits_count / domain_len) if domain_len > 0 else 0.0
    entropy = calculate_entropy(domain)

    return {
        'domain_length': domain_len,
        'num_dots': dots_count,
        'num_hyphens': hyphens_count,
        'digit_ratio': digit_ratio,
        'entropy': entropy
    }

# 2. Extract features from cleaned dataset
features_list = [extract_lexical_features(d) for d in cleaned_df['domain']]
features_df = pd.DataFrame(features_list)

# 3. Print validation metrics
print("=== Missing Values in Features ===")
print(features_df.isnull().sum())

print("=== Infinite Values in Features ===")
print(np.isinf(features_df).sum())

print("=== Descriptive Statistics of Extracted Features ===")
display(features_df.describe())

# Keep features separate but easily matchable via index
print("Metadata and feature shapes:")
print(f"cleaned_df shape: {cleaned_df.shape}")
print(f"features_df shape: {features_df.shape}")

=== Missing Values in Features ===
domain_length    0
num_dots         0
num_hyphens      0
digit_ratio      0
entropy          0
dtype: int64
=== Infinite Values in Features ===
domain_length    0
num_dots         0
num_hyphens      0
digit_ratio      0
entropy          0
dtype: int64
=== Descriptive Statistics of Extracted Features ===


,domain_length,num_dots,num_hyphens,digit_ratio,entropy
count,19945.000000,19945.000000,19945.000000,19945.000000,19945.000000
mean,19.140085,1.484081,0.374129,0.051953,3.421590
std,11.084198,0.588498,0.798058,0.117775,0.487863
min,4.000000,1.000000,0.000000,0.000000,0.755365
25%,11.000000,1.000000,0.000000,0.000000,3.095795
50%,15.000000,1.000000,0.000000,0.000000,3.418296
75%,25.000000,2.000000,1.000000,0.025000,3.784942
max,104.000000,8.000000,11.000000,0.750000,4.882922


Metadata and feature shapes:
cleaned_df shape: (19945, 4)
features_df shape: (19945, 5)


**Reasoning**:
The previous code block contained a syntax error due to an unterminated string literal when trying to print a newline and text over multiple lines. I will provide the fixed code block with correct string formatting.



In [16]:
import numpy as np
import pandas as pd
import math
from collections import Counter

# 1. Define feature extraction function
def calculate_entropy(s):
    if not s:
        return 0
    counts = Counter(s)
    length = len(s)
    return -sum((count / length) * math.log2(count / length) for count in counts.values())

def extract_lexical_features(domain):
    if not isinstance(domain, str):
        domain = ""

    domain_len = len(domain)
    dots_count = domain.count('.')
    hyphens_count = domain.count('-')
    digits_count = sum(c.isdigit() for c in domain)
    digit_ratio = (digits_count / domain_len) if domain_len > 0 else 0.0
    entropy = calculate_entropy(domain)

    return {
        'domain_length': domain_len,
        'num_dots': dots_count,
        'num_hyphens': hyphens_count,
        'digit_ratio': digit_ratio,
        'entropy': entropy
    }

# 2. Extract features from cleaned dataset
features_list = [extract_lexical_features(d) for d in cleaned_df['domain']]
features_df = pd.DataFrame(features_list)

# 3. Print validation metrics
print("=== Missing Values in Features ===")
print(features_df.isnull().sum())

print("\n=== Infinite Values in Features ===")
print(np.isinf(features_df).sum())

print("\n=== Descriptive Statistics of Extracted Features ===")
display(features_df.describe())

# Keep features separate but easily matchable via index
print("\nMetadata and feature shapes:")
print(f"cleaned_df shape: {cleaned_df.shape}")
print(f"features_df shape: {features_df.shape}")

=== Missing Values in Features ===
domain_length    0
num_dots         0
num_hyphens      0
digit_ratio      0
entropy          0
dtype: int64

=== Infinite Values in Features ===
domain_length    0
num_dots         0
num_hyphens      0
digit_ratio      0
entropy          0
dtype: int64

=== Descriptive Statistics of Extracted Features ===


,domain_length,num_dots,num_hyphens,digit_ratio,entropy
count,19945.000000,19945.000000,19945.000000,19945.000000,19945.000000
mean,19.140085,1.484081,0.374129,0.051953,3.421590
std,11.084198,0.588498,0.798058,0.117775,0.487863
min,4.000000,1.000000,0.000000,0.000000,0.755365
25%,11.000000,1.000000,0.000000,0.000000,3.095795
50%,15.000000,1.000000,0.000000,0.000000,3.418296
75%,25.000000,2.000000,1.000000,0.025000,3.784942
max,104.000000,8.000000,11.000000,0.750000,4.882922



Metadata and feature shapes:
cleaned_df shape: (19945, 4)
features_df shape: (19945, 5)


## Define Safeguarded Output Filenames

### Subtask:
Propose and save the engineered dataset and features using secure, structured, and non-overlapping filenames.


**Reasoning**:
We will concatenate the metadata and features into a single dataframe, verify its structure, save it alongside the decoupled features dataframe, and verify the resulting files.



In [18]:
import pandas as pd
import os

# 1. Concatenate metadata columns and extracted features along the column axis
integrated_df = pd.concat([cleaned_df.reset_index(drop=True), features_df.reset_index(drop=True)], axis=1)

# 2. Verify that the combined DataFrame has exactly 19,945 rows and 9 columns
print("=== Combined DataFrame Verification ===")
print(f"Integrated Shape: {integrated_df.shape} (Expected: (19945, 9))")
assert integrated_df.shape == (19945, 9), "Error: Integrated dataset shape is not exactly (19945, 9)!"

# 3. Save integrated dataset
integrated_path = '/content/cleaned_combined_domains_with_features.csv'
integrated_df.to_csv(integrated_path, index=False)
print(f"Integrated dataset saved to: {integrated_path}")

# 4. Save separate features dataset
features_path = '/content/processed_features_v2.csv'
features_df.to_csv(features_path, index=False)
print(f"Decoupled features saved to: {features_path}")

# 5. Verify files exist and can be reloaded
print("\n=== File Reload Verification ===")
test_integrated = pd.read_csv(integrated_path)
test_features = pd.read_csv(features_path)
print(f"Reloaded Integrated Shape: {test_integrated.shape}")
print(f"Reloaded Features Shape: {test_features.shape}")
assert test_integrated.shape == (19945, 9), "Failed to verify reloaded integrated file."
assert test_features.shape == (19945, 5), "Failed to verify reloaded features file."
print("All file operations verified successfully!")

=== Combined DataFrame Verification ===
Integrated Shape: (19945, 9) (Expected: (19945, 9))
Integrated dataset saved to: /content/cleaned_combined_domains_with_features.csv
Decoupled features saved to: /content/processed_features_v2.csv

=== File Reload Verification ===
Reloaded Integrated Shape: (19945, 9)
Reloaded Features Shape: (19945, 5)
All file operations verified successfully!


## Summary:

### Q&A

* **What was the outcome of the audit and exclusion step?**
  * Five legitimate platform domains (`drive.google.com`, `docs.google.com`, `firebasestorage.googleapis.com`, `www.dropbox.com`, and `linktr.ee`) were identified as false positives and separated from the training set.
  * Three malicious cloud storage subdomains were retained as malicious (label 1).
  * This reduced the dataset size from exactly $19,950$ rows to exactly $19,945$ rows.

* **What corrections were made to the source attributions?**
  * The `source` attribute for the three retained cloud storage domains was updated from `PhishTank+URLhaus` to `PhishTank` after auditing verified they did not exist in the original URLhaus database. This left $0$ records in the cleaned dataset with the dual `PhishTank+URLhaus` source label.

* **What is the final label and source distribution of the cleaned dataset?**
  * **Label Distribution:**
    * Malicious (1): $9,995$ records ($50.11\%$)
    * Benign (0): $9,950$ records ($49.89\%$)
  * **Source Distribution:**
    * Tranco: $9,950$ records
    * PhishTank: $9,019$ records
    * URLhaus: $976$ records

---

### Data Analysis Key Findings

* **Successful Feature Engineering:** Five lexical features were extracted from the domain strings to assist model training:
  * **Domain Length:** Average of ~19.14 characters.
  * **Number of Dots:** Average of ~1.48 dots per domain.
  * **Number of Hyphens:** Count of `-` characters.
  * **Digit Ratio:** Proportion of numerical digits to domain length.
  * **Shannon Entropy:** Average of ~3.42, measuring the randomness of the domain strings.
* **Data Quality and Alignment:**
  * Cleaned datasets contained $0$ missing and $0$ infinite values across all computed features.
  * The integrated dataset matching the cleaned metadata with the engineered features resulted in a verified shape of exactly $19,945$ rows and $9$ columns.

---

### Insights or Next Steps

* **Model Training Readiness:** With a highly balanced class distribution of $50.11\%$ malicious to $49.89\%$ benign and structured numerical features, the dataset is primed for training classification models (such as XGBoost, Random Forests, or Logistic Regression).
* **Next Step:** Perform feature scaling (e.g., standardizing domain length and entropy) and split the integrated dataset (`/content/cleaned_combined_domains_with_features.csv`) into training, validation, and test sets.


# Task
Perform a data quality audit and partition the preprocessed dataset in `"/content/cleaned_combined_domains_with_features.csv"`. Split the feature matrix consisting of the five lexical features (`domain_length`, `num_dots`, `num_hyphens`, `digit_ratio`, `entropy`) and the target column (`label`) into 80% training, 10% validation, and 10% testing subsets using stratified sampling with `random_state=42`. Standardize the splits using a `StandardScaler` fitted exclusively on the training set, export both scaled and unscaled splits to CSV files, verify that there are no domain overlaps, and output a detailed distribution report.

## Audit and Inspect Preprocessed Dataset

### Subtask:
Load and inspect the preprocessed dataset in '/content/cleaned_combined_domains_with_features.csv' to verify shape, values, and feature types.


**Reasoning**:
Load and audit the preprocessed dataset 'cleaned_combined_domains_with_features.csv' to verify its shape, column types, missing values, and check for constant features as instructed in the subtask.



In [20]:
import pandas as pd
import numpy as np

# 1. Load the preprocessed dataset
preprocessed_path = '/content/cleaned_combined_domains_with_features.csv'
preprocessed_df = pd.read_csv(preprocessed_path)

# 2. Output shape
print("=== Dataset Shape ===")
print(preprocessed_df.shape)
assert preprocessed_df.shape == (19945, 9), f"Expected (19945, 9), but got {preprocessed_df.shape}"

# 3. Check for missing (null) or infinite values
print("\n=== Missing Values Per Column ===")
print(preprocessed_df.isnull().sum())

# Check for infinite values in numeric columns
numeric_cols = preprocessed_df.select_dtypes(include=[np.number]).columns
print("\n=== Infinite Values in Numeric Columns ===")
for col in numeric_cols:
    inf_count = np.isinf(preprocessed_df[col]).sum()
    print(f"{col}: {inf_count}")

# 4. Categorize columns
target_col = ['label']
lexical_features = ['domain_length', 'num_dots', 'num_hyphens', 'digit_ratio', 'entropy']
metadata_cols = ['domain', 'source', 'collection_date']

print("\n=== Column Classification ===")
print(f"Target column: {target_col}")
print(f"Lexical features: {lexical_features}")
print(f"Metadata columns: {metadata_cols}")

# 5. Check for constant features (zero variance)
print("\n=== Feature Variances (Constant Check) ===")
variances = preprocessed_df[lexical_features].var()
print(variances)
constant_feats = variances[variances == 0].index.tolist()
if len(constant_feats) == 0:
    print("\nVerification Successful: No constant features found with zero variance.")
else:
    print(f"\nWARNING: Constant features detected: {constant_feats}")

=== Dataset Shape ===
(19945, 9)

=== Missing Values Per Column ===
domain             0
label              0
source             0
collection_date    0
domain_length      0
num_dots           0
num_hyphens        0
digit_ratio        0
entropy            0
dtype: int64

=== Infinite Values in Numeric Columns ===
label: 0
domain_length: 0
num_dots: 0
num_hyphens: 0
digit_ratio: 0
entropy: 0

=== Column Classification ===
Target column: ['label']
Lexical features: ['domain_length', 'num_dots', 'num_hyphens', 'digit_ratio', 'entropy']
Metadata columns: ['domain', 'source', 'collection_date']

=== Feature Variances (Constant Check) ===
domain_length    122.859437
num_dots           0.346330
num_hyphens        0.636896
digit_ratio        0.013871
entropy            0.238011
dtype: float64

Verification Successful: No constant features found with zero variance.


## Perform Stratified Train-Validation-Test Splitting

### Subtask:
Partition the preprocessed dataset into 80% training, 10% validation, and 10% testing subsets using stratified sampling based on labels.


**Reasoning**:
Perform a stratified train-validation-test split (80/10/10) on the preprocessed dataset, extract the domain splits, and verify that there is no domain overlap between the partitions.



In [22]:
from sklearn.model_selection import train_test_split

# 1. First split: 80% train, 20% temp (which will be split into val and test)
train_df, temp_df = train_test_split(
    preprocessed_df,
    test_size=0.2,
    stratify=preprocessed_df['label'],
    random_state=42
)

# 2. Second split: split the 20% temp into 50% validation and 50% test (each 10% of total)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['label'],
    random_state=42
)

# 3. Extract domain columns as sets
train_domains = set(train_df['domain'])
val_domains = set(val_df['domain'])
test_domains = set(test_df['domain'])

# 4. Verify absolute zero overlap
train_val_overlap = train_domains.intersection(val_domains)
train_test_overlap = train_domains.intersection(test_domains)
val_test_overlap = val_domains.intersection(test_domains)

print("=== Domain Overlap Verification ===")
print(f"Overlap between Train and Val: {len(train_val_overlap)}")
print(f"Overlap between Train and Test: {len(train_test_overlap)}")
print(f"Overlap between Val and Test: {len(val_test_overlap)}")

assert len(train_val_overlap) == 0, "Error: Train and Val sets overlap!"
assert len(train_test_overlap) == 0, "Error: Train and Test sets overlap!"
assert len(val_test_overlap) == 0, "Error: Val and Test sets overlap!"
print("Verification Successful: Zero domain overlap between splits verified.\n")

# 5. Print partition metrics and percentages
total_len = len(preprocessed_df)
print("=== Dataset Partition Distribution ===")
for name, df_split in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    count = len(df_split)
    percentage = (count / total_len) * 100
    print(f"{name:10} Partition: {count:5} rows ({percentage:.2f}%)")

=== Domain Overlap Verification ===
Overlap between Train and Val: 0
Overlap between Train and Test: 0
Overlap between Val and Test: 0
Verification Successful: Zero domain overlap between splits verified.

=== Dataset Partition Distribution ===
Train      Partition: 15956 rows (80.00%)
Validation Partition:  1994 rows (10.00%)
Test       Partition:  1995 rows (10.00%)


**Reasoning**:
Perform a stratified train-validation-test split (80/10/10) on the preprocessed dataset, extract the domain splits, and verify that there is no domain overlap between the partitions.



In [21]:
from sklearn.model_selection import train_test_split

# 1. First split: 80% train, 20% temp (which will be split into val and test)
train_df, temp_df = train_test_split(
    preprocessed_df,
    test_size=0.2,
    stratify=preprocessed_df['label'],
    random_state=42
)

# 2. Second split: split the 20% temp into 50% validation and 50% test (each 10% of total)
val_df, test_df = train_test_split(
    temp_df,
    test_size=0.5,
    stratify=temp_df['label'],
    random_state=42
)

# 3. Extract domain columns as sets
train_domains = set(train_df['domain'])
val_domains = set(val_df['domain'])
test_domains = set(test_df['domain'])

# 4. Verify absolute zero overlap
train_val_overlap = train_domains.intersection(val_domains)
train_test_overlap = train_domains.intersection(test_domains)
val_test_overlap = val_domains.intersection(test_domains)

print("=== Domain Overlap Verification ===")
print(f"Overlap between Train and Val: {len(train_val_overlap)}")
print(f"Overlap between Train and Test: {len(train_test_overlap)}")
print(f"Overlap between Val and Test: {len(val_test_overlap)}")

assert len(train_val_overlap) == 0, "Error: Train and Val sets overlap!"
assert len(train_test_overlap) == 0, "Error: Train and Test sets overlap!"
assert len(val_test_overlap) == 0, "Error: Val and Test sets overlap!"
print("Verification Successful: Zero domain overlap between splits verified.\n")

# 5. Print partition metrics and percentages
total_len = len(preprocessed_df)
print("=== Dataset Partition Distribution ===")
for name, df_split in [('Train', train_df), ('Validation', val_df), ('Test', test_df)]:
    count = len(df_split)
    percentage = (count / total_len) * 100
    print(f"{name:10} Partition: {count:5} rows ({percentage:.2f}%)")

=== Domain Overlap Verification ===
Overlap between Train and Val: 0
Overlap between Train and Test: 0
Overlap between Val and Test: 0
Verification Successful: Zero domain overlap between splits verified.

=== Dataset Partition Distribution ===
Train      Partition: 15956 rows (80.00%)
Validation Partition:  1994 rows (10.00%)
Test       Partition:  1995 rows (10.00%)


## Scale Features using StandardScaler

### Subtask:
Standardize the lexical features using StandardScaler fitted strictly on the training set to prevent data leakage.


**Reasoning**:
Standardize the five lexical features using a StandardScaler fitted strictly on the training set to prevent data leakage, keeping both scaled and unscaled splits.



In [23]:
from sklearn.preprocessing import StandardScaler

# 1. Instantiate the StandardScaler
scaler = StandardScaler()

# 2. Identify the five lexical features
lexical_features = ['domain_length', 'num_dots', 'num_hyphens', 'digit_ratio', 'entropy']

# 3. Separate target variables (y)
y_train = train_df['label'].values
y_val = val_df['label'].values
y_test = test_df['label'].values

# 4. Fit the scaler EXCLUSIVELY on the training features
scaler.fit(train_df[lexical_features])

# 5. Transform training, validation, and testing partitions
X_train_scaled = scaler.transform(train_df[lexical_features])
X_val_scaled = scaler.transform(val_df[lexical_features])
X_test_scaled = scaler.transform(test_df[lexical_features])

# Keep unscaled features intact as DataFrames
X_train_unscaled = train_df[lexical_features].copy()
X_val_unscaled = val_df[lexical_features].copy()
X_test_unscaled = test_df[lexical_features].copy()

# 6. Verify and output shape metrics
print("=== Feature Scaling Verification ===")
print(f"Scaled Train Features Shape: {X_train_scaled.shape}")
print(f"Scaled Val Features Shape:   {X_val_scaled.shape}")
print(f"Scaled Test Features Shape:  {X_test_scaled.shape}")
print(f"Target Train Labels Shape:   {y_train.shape}")

# Display mean and scale of the fitted scaler to verify correct calculation
print("\n=== Fitted Scaler Statistics (Training Set) ===")
for feat, mean, scale in zip(lexical_features, scaler.mean_, scaler.scale_):
    print(f"{feat:15} | Mean: {mean:8.4f} | Std Dev (Scale): {scale:8.4f}")

=== Feature Scaling Verification ===
Scaled Train Features Shape: (15956, 5)
Scaled Val Features Shape:   (1994, 5)
Scaled Test Features Shape:  (1995, 5)
Target Train Labels Shape:   (15956,)

=== Fitted Scaler Statistics (Training Set) ===
domain_length   | Mean:  19.1639 | Std Dev (Scale):  11.1521
num_dots        | Mean:   1.4848 | Std Dev (Scale):   0.5879
num_hyphens     | Mean:   0.3765 | Std Dev (Scale):   0.8007
digit_ratio     | Mean:   0.0518 | Std Dev (Scale):   0.1172
entropy         | Mean:   3.4222 | Std Dev (Scale):   0.4879


## Export splits and validation summary

### Subtask:
Export the scaled and unscaled data splits alongside labels as independent CSV files, and compute the final verification distributions.


**Reasoning**:
Export the unscaled and scaled splits with their corresponding labels to CSV files and print the final distribution of labels to verify the stratified split.



In [24]:
import pandas as pd

# 1. Convert scaled arrays into pandas DataFrames with correct feature names
lexical_features = ['domain_length', 'num_dots', 'num_hyphens', 'digit_ratio', 'entropy']

X_train_scaled_df = pd.DataFrame(X_train_scaled, columns=lexical_features, index=train_df.index)
X_val_scaled_df = pd.DataFrame(X_val_scaled, columns=lexical_features, index=val_df.index)
X_test_scaled_df = pd.DataFrame(X_test_scaled, columns=lexical_features, index=test_df.index)

# 2. Create copies of unscaled feature DataFrames to safely attach labels
train_unscaled_out = X_train_unscaled.copy()
val_unscaled_out = X_val_unscaled.copy()
test_unscaled_out = X_test_unscaled.copy()

# Attach target labels
train_unscaled_out['label'] = y_train
val_unscaled_out['label'] = y_val
test_unscaled_out['label'] = y_test

# Attach target labels to scaled DataFrames
X_train_scaled_df['label'] = y_train
X_val_scaled_df['label'] = y_val
X_test_scaled_df['label'] = y_test

# 3. Save unscaled splits to CSV
train_unscaled_out.to_csv('/content/train_unscaled.csv', index=False)
val_unscaled_out.to_csv('/content/val_unscaled.csv', index=False)
test_unscaled_out.to_csv('/content/test_unscaled.csv', index=False)

# 4. Save scaled splits to CSV
X_train_scaled_df.to_csv('/content/train_scaled.csv', index=False)
X_val_scaled_df.to_csv('/content/val_scaled.csv', index=False)
X_test_scaled_df.to_csv('/content/test_scaled.csv', index=False)

print("=== File Export Completed successfully! ===")

# 5. Print value counts of labels for verification
for name, labels in [('Train', y_train), ('Validation', y_val), ('Test', y_test)]:
    print(f"\n=== {name} Split Class Distribution ===")
    unique, counts = np.unique(labels, return_counts=True)
    dist = dict(zip(unique, counts))
    total = sum(counts)
    for cls, cnt in dist.items():
        print(f"  Class {cls}: {cnt} ({cnt/total*100:.2f}%)")

=== File Export Completed successfully! ===

=== Train Split Class Distribution ===
  Class 0: 7960 (49.89%)
  Class 1: 7996 (50.11%)

=== Validation Split Class Distribution ===
  Class 0: 995 (49.90%)
  Class 1: 999 (50.10%)

=== Test Split Class Distribution ===
  Class 0: 995 (49.87%)
  Class 1: 1000 (50.13%)


## Final Task

### Subtask:
Provide a comprehensive summary of the training-readiness audit, verify file paths, and request explicit approval before initiating model training.


## Summary:

### Q&A

* **What was the outcome of the data quality audit on the preprocessed dataset?**
  The dataset located at `/content/cleaned_combined_domains_with_features.csv` successfully passed the audit. It contains exactly 19,945 rows and 9 columns, with 0 missing (null) values and 0 infinite values. No constant features (zero variance) were detected among the lexical features.
* **How was the dataset split and was there any domain leakage?**
  The dataset was partitioned using stratified sampling (random state 42) into an 80% training set (15,956 rows), a 10% validation set (1,994 rows), and a 10% testing set (1,995 rows). Set intersection checks confirmed exactly zero domain overlaps among the splits, ensuring absolute prevention of domain leakage.
* **Where are the final exported partitions saved?**
  * **Unscaled Splits:** `/content/train_unscaled.csv`, `/content/val_unscaled.csv`, and `/content/test_unscaled.csv`
  * **Scaled Splits:** `/content/train_scaled.csv`, `/content/val_scaled.csv`, and `/content/test_scaled.csv`

---

### Data Analysis Key Findings

* **Perfect Stratification:** The target label class distributions are perfectly balanced at approximately 50% benign (Class 0) and 50% malicious (Class 1) across all partitions:
  * **Train Split:** 49.89% Benign, 50.11% Malicious
  * **Validation Split:** 49.90% Benign, 50.10% Malicious
  * **Test Split:** 49.87% Benign, 50.13% Malicious
* **Leakage-Free Standard Scaling:** The `StandardScaler` was fitted strictly on the training partition to calculate the scaling parameters, completely isolating validation and test sets from future data leakage:
  * `domain_length`: Mean = 19.1639, Std Dev = 11.1521
  * `num_dots`: Mean = 1.4848, Std Dev = 0.5879
  * `num_hyphens`: Mean = 0.3765, Std Dev = 0.8007
  * `digit_ratio`: Mean = 0.0518, Std Dev = 0.1172
  * `entropy`: Mean = 3.4222, Std Dev = 0.4879

---

### Insights or Next Steps

* **Model Training Readiness:** The dataset is fully prepared for binary classification. Use the unscaled CSV files for tree-based models (such as Random Forests or Gradient Boosting Machines) and the scaled CSV files for algorithms sensitive to feature scaling (such as Logistic Regression, Support Vector Machines, or Neural Networks).
* **Audit Approval Request:** Submit the verified file paths and partition distribution details for stakeholder approval before commencing model training.
